In [ ]:
'''Курсовая работа на тему Анализ ходов фигуры Конь - Дракон(12) на шахматной доске'''

'''Импорт нужных библиотек:'''
import tkinter
import pygame
from solver import ChessBoard

'''Создание окна для ввода N_L_K:'''
root = tkinter.Tk()
root.title('Курсовая работа "Шахматы"')
root.geometry('170x150')
root.configure(bg='white')

'''Заполнение полей ввода:'''
N, N_entry = tkinter.Label(master=root, text='Размер доски (N):'), tkinter.Entry(width=15)
N.pack()
N_entry.pack()
L, L_entry = tkinter.Label(master=root, text='Фигур на расставление (L):'), tkinter.Entry(width=15)
L.pack()
L_entry.pack()
K, K_entry = tkinter.Label(master=root, text='Стоящие фигуры (K):'), tkinter.Entry(width=15)
K.pack()
K_entry.pack()

good_value, N, L, K = False, 0, 0, 0

'''Метод от которого будет выведено сообщение об ошибке:'''


def wrong_enter():
    wrong_root = tkinter.Tk()
    wrong_root.title('Ошибка!')
    wrong_root.geometry('240x50')
    tkinter.Label(master=wrong_root, text='Неверный формат ввода!').pack()
    tkinter.Button(master=wrong_root, text='Ок', command=lambda: wrong_root.destroy()).pack()
    wrong_root.mainloop()


'''Обработка N_L_K с проверкой на правильность введенных данных:'''


def enter_data():
    global good_value, N, L, K
    N, L, K = N_entry.get(), L_entry.get(), K_entry.get()
    root.destroy()
    if not N.isdigit() or not L.isdigit() or not K.isdigit():
        wrong_enter()
    else:
        N, L, K = int(N), int(L), int(K)
        good_value = True


enter_button = tkinter.Button(text="Ввести", command=enter_data)
enter_button.pack()
root.mainloop()

'''Проверка наличия ошибок через enter_data:'''
if not good_value:
    raise TypeError('Тип данных неверен')

'''Обработка введённых координат:'''


def enter_coords_for_k(entry_pole: list):
    for c in enter_line:
        line = c.get()
        coords_x_y = tuple(line.split())
        if len(coords_x_y) == 2 and False not in tuple(map(lambda x: x.isdigit(), coords_x_y)):
            chess_pices.append((int(coords_x_y[0]), int(coords_x_y[1])))
        else:
            wrong_enter()
    k_root.destroy()


chess_pices = []
k_root = tkinter.Tk()
k_root.geometry(f'250x{20 * (K + 1) + 50}')
k_root.title('Координаты (K)')
k_root.configure(bg='white')
tkinter.Label(master=k_root, text='Введите координаты (К) через пробел').pack()
enter_line = [tkinter.Entry(master=k_root) for j in range(K)]
for i in range(K):
    enter_line[i].pack()

'''Кнопка ввода c командой на активацию функции enter_coords_for_k:'''
enter_button = tkinter.Button(text="Ввод данных", command=lambda: enter_coords_for_k(enter_line))
enter_button.pack()
k_root.mainloop()

'''Цвета, используемые на доске:'''
RED = (200, 25, 25)
Yellow = (255, 223, 0)
PURPLE = (240, 0, 255)
GREEN = (0, 255, 0)
LightSteelBlue4 = (110, 123, 139)
BLACK = {'black': (0, 0, 0)}
WHITE = {'white': (255, 255, 255)}

'''Подготовка к запуску шахматной доски:'''
chess = pygame.display.set_mode((500, 500))
pygame.display.set_caption('Шахматная доска')
chess.fill(LightSteelBlue4)
width = 2
cell_size = (500 - (N + 1) * width) / N
clock = pygame.time.Clock()

'''Класс клетки на доске:'''


class SquareCell(pygame.sprite.Sprite):
    def __init__(self, cel_size: float, color: str) -> None:
        super(SquareCell, self).__init__()
        self.surface = pygame.Surface((cel_size, cel_size))
        if color == 'white':
            self.surface.fill(WHITE['white'])
        else:
            self.surface.fill(BLACK['black'])
        self.rect = self.surface.get_rect()


'''Класс фигуры на доске:'''


class ChessFigure(pygame.sprite.Sprite):
    def __init__(self, x: int, y: int, type: str) -> None:
        self.x = x
        self.y = y
        self.type = type
        self.surface = pygame.Surface((cell_size, cell_size))
        if type == 'figure':
            self.surface.fill(RED)
            self.color = RED
        elif type == 'new figure':
            self.surface.fill(GREEN)
            self.color = GREEN
        elif type == 'X':
            self.surface.fill(Yellow)
            self.color = Yellow
        else:
            self.surface.fill(PURPLE)
            self.color = PURPLE
        self.rect = self.surface.get_rect()
        self.pygame_x_coord = width * (x + 1) + cell_size * self.x
        self.pygame_y_coord = width * (y + 1) + cell_size * self.y

    def __repr__(self):
        if self.type == 'figure' or self.type == 'new figure':
            return 'Figure'
        else:
            return 'Cell unger battle '


'''Функция для поиска клетки под боем для фигуры Конь-Дракон (Король + Слон(на 3 клетки по диагонали)):'''
def cell_under_battle(size: int, x: int, y: int):
    beat = []

    '''Клетки рядом с фигурой:'''
    for row in range(-1, 2):
        for collumn in range(-1, 2):
            if 0 <= x + row < size and 0 <= y + collumn < size and (x + row, y + collumn) not in beat:
                beat.append((x + row, y + collumn))

    '''Клетки по диагонали:'''
    step = 2
    while step <= 3:
        for row in range(x - step, x + step + 1, step * 2):
            for col in range(y - step, y + step + 1, step * 2):
                if 0 <= row < size and 0 <= col < size and (row, col) not in beat:
                    beat.append((row, col))
        step += 1
        if (x, y) in beat:
            del beat[beat.index((x, y))]
    return [(x, y), beat]


'''Визуализация шахматной доски:'''

'''Расстановка К фигур:'''

# Инициализация доски
board = ChessBoard(N)

close = False
activ_b_c = []
cords_b = []
activ_l_c = []
cords_l = []
out_good_cords = []

for a in chess_pices:
    board.add_piece(a)

for i, j in chess_pices:
    cords_b.append(list(filter(lambda x: x not in cords_b, cell_under_battle(N, i, j))))

'''Реализация расчетов:'''
good_cords = []
for x in range(N):
    for y in range(N):
        good_cords.append((x, y))
good_cords = list(set(good_cords) - set(chess_pices))
bad_cords = []
for i in range(len(cords_b)):
    bad_cords += cords_b[i][1][:]
good_cords = sorted(list(set(good_cords) - set(bad_cords)))
data_result = []
data_result_main = []

'''Функции с рекурсией для вычислений'''


def rek_speed_load(g_c, old_g_c, new_data, old_data):
    global data_result
    res = []
    for i in range(1):
        new_data.append(g_c[i])
        g_c = list(set(g_c) - set(new_data))
        a = []
        for x, y in new_data:
            a += list(filter(lambda x: x not in a, cell_under_battle(N, x, y)[1]))
        g_c = sorted(list(set(g_c) - set(a)))
        if len(new_data) == L:
            res.append(new_data)
            g_c = old_g_c.copy()
            new_data = old_data.copy()
        else:
            data_result.append(rek_speed_load(g_c.copy(), g_c.copy(), new_data.copy(), new_data.copy()))
            g_c = old_g_c.copy()
            new_data = old_data.copy()
    print('Loading...')
    return res


def rek(g_c, old_g_c, new_data, old_data):
    global data_result_main
    res = []
    for i in range(len(g_c)):
        new_data.append(g_c[i])
        g_c = list(set(g_c) - set(new_data))
        a = []
        for x, y in new_data:
            a += list(filter(lambda x: x not in a, cell_under_battle(N, x, y)[1]))
        g_c = sorted(list(set(g_c) - set(a)))
        if len(new_data) == L:

            res.append(new_data)
            g_c = old_g_c.copy()
            new_data = old_data.copy()
        else:
            data_result_main.append(rek(g_c.copy(), g_c.copy(), new_data.copy(), new_data.copy()))
            g_c = old_g_c.copy()
            new_data = old_data.copy()
    print('Loading...')
    return res


'''Поиск первого решения для визуализации'''
out_good_cords = good_cords.copy()
try:
    if L > 1:
        rek_speed_load(good_cords, good_cords, [], [])
        data_result_new = []
        for i in data_result:
            data_result_new += i
        chess_pices_l = data_result_new[0]
    else:
        data_result = rek_speed_load(good_cords, good_cords, [], [])
        chess_pices_l = data_result[0]
except:
    print('Решений не найдено')
    exit(0)

if not chess_pices_l:
    print('Решений не найдено')
    exit(0)


print(data_result)

for i, j in chess_pices_l:
    cords_l.append(list(filter(lambda x: x not in cords_l, cell_under_battle(N, i, j))))

'''Отрисовка доски:'''
while True:
    clock.tick(50)
    cell_line = []
    cell_cords_line = []
    for x in range(N):
        cell_line.append([])
        cell_cords_line.append([])
        for y in range(N):
            coord_x, coord_y = width * (x + 1) + cell_size * x, width * (y + 1) + cell_size * y
            if (x + y) % 2 == 0:
                cell_line[x].append(SquareCell(cell_size, 'white').surface)
            else:
                cell_line[x].append(SquareCell(cell_size, 'black').surface)
            cell_cords_line[x].append((coord_x, coord_y, coord_x + cell_size, coord_y + cell_size))
            chess.blit(source=cell_line[x][y], dest=(coord_x, coord_y))

    ''' Функция расчета всех ходов и сохранения'''
    def output():
        global data_result_main
        if L > 1:
            rek(out_good_cords, out_good_cords, [], [])
            data_result_new = []
            for i in data_result_main:
                data_result_new += i
            out_result = data_result_new
        else:
            data_result_main = rek(out_good_cords, out_good_cords, [], [])
            out_result = data_result_main

        with open('output.txt', 'w') as f:
            for st in out_result:
                st_res = ''
                for tup in st:
                    st_res += str(tup) + ', '
                for tup in chess_pices:
                    st_res += str(tup) + ', '
                f.write(st_res[:-2])
                f.write('\n')

    if not close:
        '''Отрисовка фигур:'''
        for i, j in chess_pices:
            chess_figure = ChessFigure(i, j, 'figure')
            chess.blit(source=chess_figure.surface, dest=(chess_figure.pygame_x_coord, chess_figure.pygame_y_coord))

        for i, j in chess_pices_l:
            chess_figure = ChessFigure(i, j, 'new figure')
            chess.blit(source=chess_figure.surface, dest=(chess_figure.pygame_x_coord, chess_figure.pygame_y_coord))

        '''Отрисовка клеток под боем:'''
        for ind_b in range(len(activ_b_c)):
            for q, w in activ_b_c[ind_b]:
                if (q, w) not in chess_pices and (q, w) not in chess_pices_l:
                    cur_cell_u_b = ChessFigure(q, w, 'cell_under_battle')
                    chess.blit(source=cur_cell_u_b.surface, dest=(cur_cell_u_b.pygame_x_coord, cur_cell_u_b.pygame_y_coord))
                else:
                    cur_cell_u_b = ChessFigure(q, w, 'X')
                    chess.blit(source=cur_cell_u_b.surface, dest=(cur_cell_u_b.pygame_x_coord, cur_cell_u_b.pygame_y_coord))

        for ind_l in range(len(activ_l_c)):
            for q, w in activ_l_c[ind_l]:
                if (q, w) not in chess_pices and (q, w) not in chess_pices_l:
                    cur_cell_u_b = ChessFigure(q, w, 'cell_under_battle')
                    chess.blit(source=cur_cell_u_b.surface, dest=(cur_cell_u_b.pygame_x_coord, cur_cell_u_b.pygame_y_coord))
                else:
                    cur_cell_u_b = ChessFigure(q, w, 'X')
                    chess.blit(source=cur_cell_u_b.surface, dest=(cur_cell_u_b.pygame_x_coord, cur_cell_u_b.pygame_y_coord))

        '''Проверка нажатий на кнопки'''
        for ev in pygame.event.get():
            if ev.type == pygame.QUIT:
                close = True
            if ev.type == pygame.MOUSEBUTTONDOWN:
                if ev.button == 1:
                    clik_x = ev.pos[0] // (500 // N)
                    clik_y = ev.pos[1] // (500 // N)
                    try:
                        if cords_b[[i[0] for i in cords_b].index((clik_x, clik_y))][1] in activ_b_c:
                            del activ_b_c[activ_b_c.index(cords_b[[i[0] for i in cords_b].index((clik_x, clik_y))][1])]
                        else:
                            activ_b_c.append(cords_b[[i[0] for i in cords_b].index((clik_x, clik_y))][1])
                    except Exception as ex:
                        pass
                    try:
                        if cords_l[[i[0] for i in cords_l].index((clik_x, clik_y))][1] in activ_l_c:
                            del activ_l_c[activ_l_c.index(cords_l[[i[0] for i in cords_l].index((clik_x, clik_y))][1])]
                        else:
                            activ_l_c.append(cords_l[[i[0] for i in cords_l].index((clik_x, clik_y))][1])
                    except Exception as ex:
                        pass
                if ev.button == 3:
                    output()
                    close = True

        pygame.display.flip()
    if close:
        break
